In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
df = pd.read_csv("dormmatch_cleaned.csv")

In [ ]:
feature_cols = ["sleep_duration","cleanliness_level","noise_tolerance","study_hours",
                "social_activity","introvert_extrovert","guest_frequency","music_frequency"]
X = df[feature_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
scores = []
for k in range(2,11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    scores.append({"k":k, "inertia":model.inertia_, "silhouette":silhouette_score(X_scaled,labels)})
scores_df = pd.DataFrame(scores)
scores_df

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(scores_df["k"], scores_df["inertia"], marker="o")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(scores_df["k"], scores_df["silhouette"], marker="o")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score by K")
plt.show()

In [ ]:
best_k = int(scores_df.loc[scores_df["silhouette"].idxmax(),"k"])
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
best_k, df["cluster"].value_counts().sort_index()

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["pca_1"] = pca_coords[:,0]
df["pca_2"] = pca_coords[:,1]
plt.figure(figsize=(9,6))
sns.scatterplot(data=df, x="pca_1", y="pca_2", hue="cluster", palette="tab10")
plt.title("Lifestyle Clusters - PCA Projection")
plt.tight_layout()
plt.show()

In [ ]:
cluster_profiles = df.groupby("cluster")[feature_cols].mean().round(2)
cluster_profiles

In [ ]:
cluster_sizes = df["cluster"].value_counts().sort_index()
cluster_sizes

In [ ]:
for col in feature_cols:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x="cluster", y=col)
    plt.title(f"{col} by Cluster")
    plt.tight_layout()
    plt.show()

In [ ]:
df.to_csv("dormmatch_clustered.csv", index=False)
cluster_profiles.to_csv("dormmatch_cluster_profiles.csv")